## Collecting Steam Data with APIs
**CSC366 Final Project: What Makes Players Recommend an Indie Game?**

Data sources (no API key needed):
 * Steam Search (scraped with BeautifulSoup): list of Indie games and their review scores: https://store.steampowered.com/search/
 * Steam Store API: details for one game: https://store.steampowered.com/api/appdetails
 * Steam Reviews API: reviews for one game: https://store.steampowered.com/appreviews/
 * SteamSpy API: estimated number of owners for one game: https://steamspy.com/api.php

**How we decide a game is Indie:** a game must be Indie in **two** ways:
1. **Player tag**: players voted the "Indie" tag onto the game (we search Steam with the Indie tag)
2. **Developer genre**: the developer listed "Indie" as one of the game's genres (we check this in Step 15)

In [ ]:
from bs4 import BeautifulSoup
import requests
import json
import pandas as pd
import re
import time
from datetime import datetime

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

folder_path = '/content/drive/My Drive/CSC366/Data Science Project/'

os.makedirs(folder_path, exist_ok=True)

## Step 3: Define the Base URLs of the APIs
We also send a `User-Agent` header with every request so Steam treats us like a normal web browser. Without it, Steam sometimes sends back an **empty** answer, which causes a `JSONDecodeError`.

In [ ]:
store_url = "https://store.steampowered.com/api/appdetails"
reviews_url = "https://api.steampowered.com/IUserReviewsService/GetAppReviews/v1/"
steamspy_url = "https://steamspy.com/api.php"
search_url = "https://store.steampowered.com/search/results/"

headers = {"User-Agent": "Mozilla/5.0"}

## Step 4: Make a Simple API Request and Check the Response
We'll test with Stardew Valley. Its App ID is **413150** (Can find any game's App ID in its store page link: store.steampowered.com/app/**413150**/...)

* `appids=` tells the API which game we want
* `cc=us` tells the API to show US prices

In [ ]:
app_id = 413150

url = f"{store_url}?appids={app_id}&cc=us"
print(url)

response = requests.get(url, headers=headers)
response

## Step 5: Check if the request was successful


In [ ]:
print("Status Code:", response.status_code)

if response.status_code == 200:
    print(" Request successful! We got a response from the API.")
else:
    print(" Request failed. Please check the URL or your internet connection.")

## Step 6: Convert the JSON response to a Python dictionary

The game's information is inside `data["413150"]["data"]`. The response is very long, so only prints its keys.

In [ ]:
if response.status_code == 200:
    data = response.json()

game = data[str(app_id)]["data"]
print(game.keys())

## Step 7: Pretty print the JSON data


In [ ]:

print(json.dumps(game["price_overview"], indent=2))
print(json.dumps(game["genres"], indent=2))
print(json.dumps(game["release_date"], indent=2))

## Step 8: Extract and display specific information

In [ ]:
print("Name:", game["name"])
print("Price:", game["price_overview"]["initial"] / 100)
print("Release date:", game["release_date"]["date"])
print("Developer:", game["developers"][0])

for genre in game["genres"]:
    print("Genre:", genre["description"])

## Step 9: Write a function to get any game's details


In [ ]:
def get_game_details(app_id):

    # Construct the full URL for the game
    url = f"{store_url}?appids={app_id}&cc=us"

    # Send a GET request to the API
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed : {response.status_code}")
        return None

    # Sometimes Steam sends back an empty answer when we ask too fast
    if response.text == "" or response.text == "null":
        print("Empty response, skipping")
        return None

    data = response.json()

    # Skip DLC, bundles, and games removed from the store
    if data[str(app_id)]["success"] == False:
        print("Not a game on the store, skipping")
        return None

    game = data[str(app_id)]["data"]

    # Free games don't have a price
    if "price_overview" in game:
        price = game["price_overview"]["initial"] / 100
    else:
        price = 0

    # Put all the genres into one string, like "Indie, RPG"
    genre_list = []
    if "genres" in game:
        for genre in game["genres"]:
            genre_list.append(genre["description"])
    genres = ", ".join(genre_list)

    game_info = {
        "app_id": app_id,
        "name": game["name"],
        "type": game["type"],
        "is_free": game["is_free"],
        "price": price,
        "genres": genres,
        "release_date": game["release_date"]["date"],
    }
    return game_info

# Fetch info about Example game Stardew Valley
game_info = get_game_details(413150)
game_info

## Step 10: Make a request to the Reviews API
Settings in the link:
* `json=1` = give us JSON
* `language=english` = English reviews only
* `filter=recent` = newest reviews first
* `num_per_page=100` = up to 100 reviews (the most we can get in one request)
* `purchase_type=all` = reviews from all buyers (not only Steam purchases)
* `cursor=*` = start at the first page of reviews

If the answer is empty, it waits 5 seconds and tries a simpler link.

The response has:
* `query_summary`: totals for the whole game (total positive, total negative, rating)
* `reviews`: the list of reviews

In [ ]:
settings = {
    "appid": int(app_id),
    "filter": 1,               # 1 = newest reviews first
    "languages": ["english"],  # English reviews only
    "purchase_type": 1,        # 1 = all buyers
    "num_per_page": 100,       # up to 100 reviews
    "cursor": "*"              # start at the first page
}
url = f"{reviews_url}?input_json={json.dumps(settings)}"

response = requests.get(url, headers=headers)
print("Status Code:", response.status_code)

if response.text.startswith("{"):
    data = response.json()["response"]   # everything is inside "response"
    print(data.keys())
    print(json.dumps(data["query_summary"], indent=2))
else:
    print("Not JSON:", response.text[:200])

In [ ]:
# Looks at the first review
if response.text != "" and len(data["reviews"]) > 0:
    first_review = data["reviews"][0]

    print("Recommended:", first_review["voted_up"])
    print("Playtime (minutes):", first_review["author"]["playtime_at_review"])
    print("Review:", first_review["review"])

## Step 11: Write a function to get a game's reviews
It returns two things:
* `summary`: the totals for the game
* `review_list`: a list of reviews, one dictionary per review

We do **not** save reviewer usernames or Steam IDs (privacy).

In [ ]:
def get_reviews(app_id):

    settings = {
        "appid": int(app_id),
        "filter": 1,
        "languages": ["english"],
        "purchase_type": 1,
        "num_per_page": 100,
        "cursor": "*"
    }
    url = f"{reviews_url}?input_json={json.dumps(settings)}"
    response = requests.get(url, headers=headers)

    if response.status_code != 200 or not response.text.startswith("{"):
        print(f"Reviews failed : {response.status_code}")
        return None, []

    data = response.json()["response"]

    # games with no reviews have no summary or review list
    if "query_summary" not in data or "reviews" not in data:
        return None, []

    summary = data["query_summary"]

    review_list = []
    for review in data["reviews"]:
        review_list.append({
            "app_id": app_id,
            "review_id": review["recommendationid"],
            "recommended": review["voted_up"],
            "review_text": review["review"],
            "playtime_at_review": review["author"]["playtime_at_review"],
            "timestamp_created": review["timestamp_created"],
            "votes_helpful": review["votes_up"],
        })

    return summary, review_list

summary, review_list = get_reviews(413150)
print(summary)
print(len(review_list), "reviews")

df = pd.DataFrame(review_list)
df.head()

## Step 12: Write a function to get the number of owners from SteamSpy
SteamSpy estimates how many people own each game (as a range, like "1,000,000 .. 2,000,000").

SteamSpy sometimes sends back an **empty** answer even with status 200. Calling `.json()` on an empty answer causes a `JSONDecodeError`, so we check `response.text` first and use `None` (missing) if it's empty.

In [ ]:
def get_owners(app_id):
    """
    Makes a GET request to the SteamSpy API using the game's App ID
    and returns the estimated number of owners (or None if it fails).
    """
    url = f"{steamspy_url}?request=appdetails&appid={app_id}"
    response = requests.get(url, headers=headers)

    if response.status_code == 200 and response.text != "":
        data = response.json()
        return data["owners"]
    else:
        return None

print(get_owners(413150))

## Step 13: Scrape the list of Indie games from Steam's search page
This is the same search page as https://store.steampowered.com/search/?tags=492 (492 is the player-voted Indie tag).

Settings in the link:
* `tags=492` = games with the Indie tag only
* `category1=998` = games only (no DLC or soundtracks)
* `start=` / `count=100` = which results to get (100 at a time)
* `infinite=1` = send the results back as JSON

The JSON has a key called `results_html` that holds the HTML of the search results. We parse it with BeautifulSoup:
* each game is an `<a>` tag with class `search_result_row`
* its App ID is in the attribute `data-ds-appid`
* its name is in `<span class="title">`
* its review score is in a `<span>` with class `search_review_summary`. Its `data-tooltip-html` attribute has text like *"Very Positive<br>94% of the 12,345 user reviews for this game are positive."* We pull out the **94** (percent positive) and **12,345** (number of reviews) with a regular expression.

Steam lists popular games first, so the first pages are almost all well-reviewed games. That's why we go **50 pages deep (up to 5,000 games)**: the deeper pages have less popular games, including poorly reviewed ones. Games with no review score yet are skipped.

In [ ]:
app_ids = []
names = []
pct_positives = []
review_counts = []

for page in range(50):
    start = page * 100
    url = f"{search_url}?query&start={start}&count=100&tags=492&category1=998&infinite=1"
    response = requests.get(url, headers=headers)

    if response.status_code != 200 or response.text == "":
        print(f"Failed : {response.status_code}")
        break

    data = response.json()
    soup = BeautifulSoup(data["results_html"], "html.parser")

    for game in soup.find_all("a", class_="search_result_row"):
        # bundles have more than one App ID like "123,456" -> skip them
        if "," in game["data-ds-appid"]:
            continue

        # games with no reviews yet have no review score -> skip them
        review_span = game.find("span", class_="search_review_summary")
        if review_span == None:
            continue

        # example tooltip: "Very Positive<br>94% of the 12,345 user reviews for this game are positive."
        tooltip = review_span["data-tooltip-html"]
        match = re.search(r"(\d+)% of the ([\d,]+) user reviews", tooltip)
        if match == None:
            continue

        app_ids.append(int(game["data-ds-appid"]))
        names.append(game.find("span", class_="title").text)
        pct_positives.append(int(match.group(1)))
        review_counts.append(int(match.group(2).replace(",", "")))

    print("Page", page + 1, "-", len(app_ids), "games so far")
    time.sleep(2)

indie_df = pd.DataFrame({
    "appid": app_ids,
    "name": names,
    "pct_positive": pct_positives,
    "total": review_counts
})
indie_df = indie_df.drop_duplicates(subset="appid")

# keep games with at least 50 reviews (too few to study otherwise)
indie_df = indie_df[indie_df["total"] >= 50]
print(len(indie_df), "Indie-tagged games with 50+ reviews")
indie_df.head()

## Step 14: Choose our sample (stratified by rating)
If we picked games completely at random, we would still get mostly positive games, because most games on Steam are positively reviewed. So we use a **stratified sample**:
1. Put each game into a rating group, using the same cutoffs Steam uses for its ratings
2. Randomly pick games from **each group**

We want **40 games per group** (about 160 games). But some games will be skipped in Step 15 (not listed as Indie by the developer, DLC, or fewer than 50 English reviews), so we pick **70 per group** as backups. Step 15 stops taking games from a group once it has 40.

* If a group has fewer than 70 games, we take all of them.
* `random_state=42` makes it pick the same games every time we run it.

**Note:** Because each group gets the same number of games, our sample has more poorly reviewed games than Steam really does. That's on purpose, but it means our overall "percent recommended" is not typical of Steam as a whole.

In [ ]:
indie_df["rating_group"] = pd.cut(
    indie_df["pct_positive"],
    bins=[0, 40, 70, 80, 101],
    labels=["Negative (under 40%)", "Mixed (40-69%)",
            "Mostly Positive (70-79%)", "Positive (80%+)"],
    include_lowest=True,
    right=False
)

print("Games in each rating group:")
print(indie_df["rating_group"].value_counts())

per_group = 40   # games we want to keep from each group
backups = 70     # games we pick from each group, in case some get skipped

sample_parts = []
for group_name, group in indie_df.groupby("rating_group", observed=True):
    n = min(backups, len(group))   # take all games if the group is small
    sample_parts.append(group.sample(n=n, random_state=42))

# shuffle so the groups are mixed together in the loop
sample_df = pd.concat(sample_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nGames picked from each group:")
print(sample_df["rating_group"].value_counts())
sample_df

## Step 15: Loop through every game in our sample
For each game we:
1. skip it if its rating group already has 40 games
2. get the game details (Steam Store API)
3. **skip it if the developer did not list "Indie" as a genre** (so every game is Indie by both the player tag and the developer)
4. get the reviews (Steam Reviews API)
5. skip the game if it has fewer than 50 English reviews (too few to study)
6. get the number of owners (SteamSpy API)
7. wait 2 seconds (`time.sleep`) between requests so we don't send them too fast

Every 25 games we save our progress to Drive, so if Colab disconnects we don't lose everything.


In [ ]:
games_list = []
reviews_list = []
count = 0

# how many games we have kept from each rating group so far
kept = {}
for group_name in sample_df["rating_group"].unique():
    kept[group_name] = 0

for index, row in sample_df.iterrows():
    app_id = row["appid"]
    group_name = row["rating_group"]

    # this group already has enough games
    if kept[group_name] >= per_group:
        continue

    count = count + 1
    print(count, "-", row["name"])

    game_info = get_game_details(app_id)
    time.sleep(2)

    # skip DLC or failed requests
    if game_info == None:
        continue

    # skip games the developer did not list as Indie
    if "Indie" not in game_info["genres"].split(", "):
        print("developer did not list Indie as a genre, skipping")
        continue

    summary, review_list = get_reviews(app_id)
    time.sleep(2)

    if summary == None:
        # reviews didn't come back: keep the game, leave review totals missing
        game_info["review_score"] = None
        game_info["total_positive"] = None
        game_info["total_negative"] = None
    else:
        # skip games with fewer than 50 English reviews
        total_reviews = summary["total_positive"] + summary["total_negative"]
        if total_reviews < 50:
            print("fewer than 50 reviews, skipping")
            continue

        # add the review totals
        game_info["review_score"] = summary["review_score_desc"]
        game_info["total_positive"] = summary["total_positive"]
        game_info["total_negative"] = summary["total_negative"]

    # add the rating group this game was sampled from
    game_info["rating_group"] = group_name

    # add the SteamSpy owners estimate
    game_info["owners"] = get_owners(app_id)
    time.sleep(2)

    games_list.append(game_info)
    for review in review_list:
        reviews_list.append(review)
    kept[group_name] = kept[group_name] + 1

    # save progress every 25 games
    if count % 25 == 0:
        pd.DataFrame(games_list).to_csv(folder_path + "games_raw.csv", index=False)
        pd.DataFrame(reviews_list).to_csv(folder_path + "reviews_raw.csv", index=False)
        print("progress saved")

print("Done!")
print(len(games_list), "games")
print(len(reviews_list), "reviews")
print("Games kept from each rating group:", kept)

## Step 16: Saving the data to CSV files
* `games_raw.csv`: one row per game
* `reviews_raw.csv`: one row per review
* `collection_log.txt`: when we collected the data and from which APIs (our provenance record)

We call it **raw** because we haven't cleaned it yet.

In [ ]:
games_df = pd.DataFrame(games_list)
reviews_df = pd.DataFrame(reviews_list)

games_df.to_csv(folder_path + "games_raw.csv", index=False)
reviews_df.to_csv(folder_path + "reviews_raw.csv", index=False)

file = open(folder_path + "collection_log.txt", "a")
file.write(str(datetime.now()) + " - Steam search scraped with BeautifulSoup (tags=492 Indie, category1=998, 50 pages, 50+ reviews, stratified 40 per rating group, random_state=42), developer genre includes Indie, Steam appdetails (cc=us), Steam appreviews (english, recent, purchase_type=all, 100 per game), SteamSpy appdetails - "
           + str(len(games_df)) + " games, " + str(len(reviews_df)) + " reviews\n")
file.close()

print("Saved to", folder_path)
games_df.head()

## Step 17: First look at missing values
The Pandas .isna() function finds missing values, and .sum() counts them.

In [ ]:
missing_values_counts = games_df.isna().sum()
missing_values_counts

In [ ]:
missing_values_percentages = missing_values_counts / len(games_df) * 100
missing_values_percentages

In [ ]:
reviews_df.isna().sum()

Saves CSV TO COMPUTER

In [ ]:
from google.colab import files

files.download(folder_path + "games_raw.csv")
files.download(folder_path + "reviews_raw.csv")